# Why the numbers are all the same

Six branches landed inside 0.8 points of each other, and the reason is in the calibration log rather than in any of the losses. On the training set the widest width reaches a loss of 0.002 and an error of 0.000: it has memorized it, while reading 75.3 percent on validation. A gap of 25 points.

That width is the teacher. At temperature 1 its soft target has been one-hot in all but name since somewhere around epoch 50.

Everything this project argues about, KL against alpha-divergence against transport against a symmetrized KL, is a question of how two distributions are compared. Against a one-hot target every divergence is the same divergence. The loss has nothing left to disagree about, so the branches converge: the gap between A and B was 1.1 points at epoch 50 and 0.5 by epoch 99.

The one place differences survive is the narrow end, which is also the only place the model has not memorized. Training error at width 0.25 is 0.035, against 0.001 at 1.00.

## The two runs

Temperature decides whether the teacher is still a distribution. These two complete a square with A and F, both of which ran at temperature 1.

| | T = 1 | T = 4 |
|---|---|---|
| KL only | A, 73.52 | **O** |
| KL and a symmetric horizontal term | F, 73.54 | **P** |

**O** says whether temperature alone helps, which is the control any claim about P needs.

**P** says whether it widens the gap the horizontal term opened. If the diagnosis holds, F beat A by 0.02 on accuracy and 0.106 on NLL while having almost no signal to work with, and both should grow. If O and P move together, the diagnosis is wrong and the losses really are interchangeable in this setting.

Read the NLL as carefully as the accuracy. Temperature acts directly on confidence, and confidence is where the only measured effect so far has been.

Set the accelerator to **GPU T4 x2**. One branch per card. Everything is reported against A, at a mean of 73.52 over sixteen widths.


In [ ]:
BRANCHES = ['o_temp4', 'p_jeffreys_temp4']

SMOKE_FIRST = True

REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'

CIFAR_DIR = ('/kaggle/input/datasets/nlnk1607/cifar100/cifar-100-python')

# To carry a timed-out session forward, attach its output and name the
# logs directory. Leave empty to start fresh.
RESUME_FROM = ''

In [ ]:
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
for i in range(n_gpu):
    print('  {}: {}'.format(i, torch.cuda.get_device_properties(i).name))

WORK = '/kaggle/working'
CODE = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(CODE):
    subprocess.run(
        ['git', 'clone', '-b', REPO_BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)

available = sorted(
    name[len('cifar100_'):-len('.yml')]
    for name in os.listdir('apps')
    if name.startswith('cifar100_') and name.endswith('.yml'))
print('\nbranches in apps/:')
for name in available:
    print('   ', name)

missing = [b for b in BRANCHES if b not in available]
if missing:
    raise SystemExit('no config for {}'.format(missing))
if n_gpu < len(BRANCHES):
    print('\n{} branches, {} gpu(s): they will run in sequence.'.format(
        len(BRANCHES), n_gpu))

In [ ]:
# What the chosen branches actually differ in, read off the configs
# rather than from the table above, which can drift.
AXES = ('kd_loss', 'cost_source', 'feature_kd', 'feature_align',
        'feature_layers', 'feature_weight', 'tier_weights',
        'horizontal_kd', 'horizontal_where', 'horizontal_loss',
        'horizontal_weight', 'weight_schedule')

settings = {}
for branch in BRANCHES:
    found = {}
    with open('apps/cifar100_{}.yml'.format(branch)) as handle:
        for line in handle:
            key = line.split(':')[0].strip()
            if key in AXES:
                found[key] = line.split(':', 1)[1].strip()
    settings[branch] = found

print('{:20}'.format('') + ''.join(
    '{:>26}'.format(b) for b in BRANCHES))
for axis in AXES:
    values = [settings[b].get(axis, '-') for b in BRANCHES]
    if all(v == '-' for v in values):
        continue
    print('{:20}'.format(axis) + ''.join(
        '{:>26}'.format(v) for v in values))

In [ ]:
TARGET = 'data/cifar-100-python'
if not os.path.isdir(TARGET):
    source = CIFAR_DIR if os.path.isdir(CIFAR_DIR) else None
    if source is None:
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'cifar-100-python' in dirs:
                source = os.path.join(root, 'cifar-100-python')
                break
    os.makedirs('data', exist_ok=True)
    if source:
        os.symlink(source, TARGET)
        print('linked', source)
    else:
        from torchvision import datasets
        datasets.CIFAR100(root='data', train=True, download=True)
        datasets.CIFAR100(root='data', train=False, download=True)
print(sorted(os.listdir(TARGET)))

## Checks, before a card is touched

Seconds on the CPU. Between them these suites have caught a Sinkhorn solved too loosely to have a correct gradient, an alpha-divergence that destroyed the weights in three steps, a calibration that reset the batch norm statistics and never refilled them, and a feature cost normalized so that its own gradient vanished. Every one of those was silent.

The last suite builds each branch in this notebook and runs two training steps of the real loop, profiling included. Four feature branches once reached Kaggle, passed every loss check, and died in the profiler on the first forward, because nothing local had ever called it.


In [ ]:
!python tests/test_loss_ops.py && python tests/test_bn_calibration.py && python tests/test_kd_variants.py && python tests/test_all_branches.py o_temp4 p_jeffreys_temp4


In [ ]:
if RESUME_FROM:
    os.makedirs('logs', exist_ok=True)
    for name in os.listdir(RESUME_FROM):
        src = os.path.join(RESUME_FROM, name)
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join('logs', name),
                            dirs_exist_ok=True)
            print('restored', name)
else:
    print('starting from scratch')

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs, quiet=True):
    """one job per card, output interleaved and tagged"""
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for index, (label, config) in enumerate(jobs):
        env = dict(os.environ)
        env['CUDA_VISIBLE_DEVICES'] = str(index % max(n_gpu, 1))
        proc = subprocess.Popen(
            [sys.executable, '-u', 'train.py', 'app:' + config],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, env=env)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] started on gpu {} with {}'.format(
            label, env['CUDA_VISIBLE_DEVICES'], config), flush=True)

    started = time.time()
    remaining = len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[float(width)] = (
                float(loss), float(top1))
        if quiet and (line.startswith(('  ', ')', 'Model(', 'Total', 'Item'))
                      or not line.strip()):
            continue
        print('[{}] {}'.format(label, line), flush=True)

    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
if SMOKE_FIRST:
    codes = run_pinned(
        [(b, 'apps/smoke_{}.yml'.format(b)) for b in BRANCHES])
    failed = [b for b, code in codes.items() if code != 0]
    if failed:
        raise SystemExit('smoke failed for {}'.format(failed))
    for b in BRANCHES:
        shutil.rmtree('logs/smoke_{}'.format(b), ignore_errors=True)
    results.clear()
    print('\nsmoke ok')

In [ ]:
codes = run_pinned(
    [(b, 'apps/cifar100_{}.yml'.format(b)) for b in BRANCHES])
print(codes)

## Results, against A

A is the number to beat, not C or D. Improving on an ablation of your own
method is not improving on the paper.

One seed, and sigma has not been measured. Three of the four gaps in the
finished table sit between 0.25 and 0.44 points, which is the range where
a single run cannot tell a result from noise.

In [ ]:
# the published run, for reference
A_KL = {0.25: 70.10, 0.30: 70.80, 0.35: 71.50, 0.40: 72.20, 0.45: 72.80,
        0.50: 73.20, 0.55: 73.40, 0.60: 73.80, 0.65: 73.90, 0.70: 74.30,
        0.75: 74.60, 0.80: 74.80, 0.85: 75.10, 0.90: 75.10, 0.95: 75.40,
        1.00: 75.30}

widths = sorted({w for table in results.values() for w in table})
header = '{:>7}{:>9}'.format('width', 'A')
for branch in BRANCHES:
    header += '{:>11}{:>8}'.format(branch[:10], 'vs A')
print(header)

for width in widths:
    row = '{:>7.2f}{:>9.2f}'.format(width, A_KL.get(width, float('nan')))
    for branch in BRANCHES:
        entry = results.get(branch, {}).get(width)
        if entry is None:
            row += '{:>11}{:>8}'.format('-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>11.2f}{:>+8.2f}'.format(
            accuracy, accuracy - A_KL.get(width, accuracy))
    print(row)

print()
reference = sum(A_KL.values()) / len(A_KL)
print('{:22} mean {:.2f}   worst {:.2f}'.format(
    'A (reference)', reference, min(A_KL.values())))
for branch in BRANCHES:
    table = results.get(branch, {})
    if not table:
        continue
    accuracies = [100.0 * (1.0 - v[1]) for v in table.values()]
    mean = sum(accuracies) / len(accuracies)
    print('{:22} mean {:.2f}   worst {:.2f}   vs A {:+.2f}'.format(
        branch, mean, min(accuracies), mean - reference))

out = os.path.join(WORK, 'logs')
for branch in BRANCHES:
    log_dir = 'logs/cifar100_{}'.format(branch)
    if os.path.isdir(log_dir):
        shutil.copytree(log_dir, os.path.join(out, 'cifar100_' + branch),
                        dirs_exist_ok=True)
print('\ncheckpoints copied to', out)